In [1]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath('..'))

from src.config import SolverConfig
from src.models.spacecraft import Spacecraft
from src.models.body import Body
from src.models.orbit import Orbit_Eq
from src.optimizer import Optimizer
from src.utils import cart2eq

In [2]:
from src.planet_propagator import earth_state, mars_state
Earth = Body(mu=3.986e14, state=earth_state)
Mars = Body(mu=4.283e13, state=mars_state)
Sun = Body(mu=1.327e20)

Psyche = Spacecraft()

In [ ]:
import numpy as np

# =========================================================
# TIME SETUP
# =========================================================
t0 = 0.0
TOF_guess = 180 * 24 * 3600   # initial guess: 180 days
tf_guess = t0 + TOF_guess

# =========================================================
# HELIOCENTRIC INITIAL STATE (Earth departure)
# =========================================================

r_E, v_E = Earth.state(t0)

# Earth parking orbit (simple circular approx)
r_parking_E = 6678e3
v_parking_E = np.sqrt(Earth.mu / r_parking_E)

r_rel_E = np.array([r_parking_E, 0, 0])
v_rel_E = np.array([0, v_parking_E, 0])

# rotate into heliocentric frame (aligned for now; refine later if needed)
r0 = r_E + r_rel_E
v0 = v_E + v_rel_E

m0 = Psyche.m0

y0 = np.hstack((r0, v0, m0))

# =========================================================
# TARGET STATE (Mars arrival)
# =========================================================

r_M, v_M = Mars.state(tf_guess)

r_parking_M = 3800e3
v_parking_M = np.sqrt(Mars.mu / r_parking_M)

r_rel_M = np.array([r_parking_M, 0, 0])
v_rel_M = np.array([0, v_parking_M, 0])

r_target = r_M + r_rel_M
v_target = v_M + v_rel_M

# If your optimizer uses MEE internally:
mee_target = cart2eq(np.hstack((r_target,v_target)))[0:5]
print(mee_target)


cfg = SolverConfig()
opt = Optimizer(cfg, Psyche, Earth, target_orbit=mee_target)
opt.qlaw.set_target()

def run_trajectory(tf):
    """
    Evaluate trajectory for a given final time (this is what makes it optimal-time).
    """

    sol = opt.propagator.forward(
        y0=y0,
        tspan=(t0, tf),
        control=opt.qlaw_control()
    )

    return sol

# =========================================================
# SIMPLE OUTER LOOP (MINIMUM TIME SEARCH)
# =========================================================
# (replace with real optimizer later: CMA-ES / SQP / IPOPT)

tf_values = np.linspace(120, 300, 10) * 24 * 3600  # 120–300 days
best_tf = None
best_cost = 1e99
best_sol = None

for tf in tf_values:

    sol = run_trajectory(tf)

    yf = sol.y[:, -1]

    rf = yf[0:3]
    vf = yf[3:6]

    # terminal error (position + velocity mismatch)
    pos_err = np.linalg.norm(rf - r_target)
    vel_err = np.linalg.norm(vf - v_target)

    mass = yf[6]

    cost = pos_err + 1e3 * vel_err - 1e-3 * mass + 1e-6 * tf

    if cost < best_cost:
        best_cost = cost
        best_tf = tf
        best_sol = sol

print("Best TOF [days]:", best_tf / 86400)
print("Best cost:", best_cost)

# =========================================================
# RESULT
# =========================================================

sol = best_sol
print("Final state:", sol.y[:, -1])

[inf, np.float64(-819.37014273652), np.float64(385.74930154852433), np.float64(0.016217350114983035), np.float64(-3.015118010283321e-05)]


TypeError: unsupported operand type(s) for *: 'NoneType' and 'float'